# Exhibition interpolation — gh0st_flux_lora_v2

Assembles Screen A and Screen B exhibition videos from the FLUX-generated stills
produced by `generate_exhibition.py`.

Outputs **raw 1024×1024 square video** — no text or scan line overlay.
Run `python app.py --video <mp4> --export` locally to produce the final
portrait-format exhibition MP4s (1080×1920, black background, scan line, text panel).

**Screen A:** stable → ambiguous → glitch → extreme → synthetic  
**Screen B:** extreme → synthetic → stable → ambiguous → glitch

**Sequence structure per category (N styles):** 3N−2 stills  
pure → 030mashup → 070mashup → pure → … → pure  
plus 2 boundary mashups between each category pair.

**Timing:** each still held for `HOLD_FRAMES`, then `MORPH_FRAMES` of optical flow to next.  
Default: 48+48 frames @ 24fps = 2s hold + 2s morph ≈ **10 min total**.

**Before running:** upload the generated stills to Drive:
```
Local:  spikes/flux_lora_training/output/gh0st_exhibition_v1/
Drive:  Gh0st in the Loop/outputs/gh0st_exhibition_v1/
```

In [ ]:
import os

!pip install opencv-python-headless imageio imageio-ffmpeg -q
!pip install -q rembg[cpu] onnxruntime "numpy>=1.26,<2"

from google.colab import drive
drive.mount('/content/drive')

if os.path.exists(repo_dir := '/content/gh0st-in-the-l00p'):
    !git -C {repo_dir} pull --quiet
else:
    !git clone --quiet https://github.com/jasonr2048/gh0st-in-the-l00p.git {repo_dir}

%cd {repo_dir}
print('Ready.')

In [ ]:
from datetime import datetime
from pathlib import Path
import re, json

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT    = Path('/content/drive/MyDrive/Gh0st in the Loop')
STILLS_BASE   = DRIVE_ROOT / 'outputs' / 'gh0st_exhibition_v1'
SELECTION     = Path('spikes/flux_lora_training/exhibition_source/selection_v1.txt')
OUTPUT_DIR    = DRIVE_ROOT / 'outputs'

# ── Screen sequences ──────────────────────────────────────────────────────────
SCREEN_A = ['stable', 'ambiguous', 'glitch', 'extreme', 'synthetic']
SCREEN_B = ['extreme', 'synthetic', 'stable', 'ambiguous', 'glitch']

# ── Timing ────────────────────────────────────────────────────────────────────
FPS          = 24
HOLD_FRAMES  = 48    # frames each still is held static  (48 = 2s)
MORPH_FRAMES = 48    # optical flow frames to next still  (48 = 2s)
# Total per still = HOLD_FRAMES + MORPH_FRAMES = 4s
# 154 stills × 4s ≈ 10 min (last still has no morph, but close enough)

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_SIZE  = (1024, 1024)   # (width, height) — raw square; app.py handles portrait layout

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_A = OUTPUT_DIR / f'exhibition_screen_A_{timestamp}.mp4'
OUT_B = OUTPUT_DIR / f'exhibition_screen_B_{timestamp}.mp4'

print(f'Screen A → {OUT_A.name}')
print(f'Screen B → {OUT_B.name}')
print(f'Hold: {HOLD_FRAMES/FPS:.1f}s  Morph: {MORPH_FRAMES/FPS:.1f}s  Per still: {(HOLD_FRAMES+MORPH_FRAMES)/FPS:.1f}s')


In [ ]:
# ── [Optional] Background removal review ────────────────────────────────────
# Scans all generated stills for images that have a visible background/frame.
# Only saves images where rembg removed significant background (>3% of pixels).
# Corner-dark images (already clean black bg) are skipped without running rembg.
#
# Output: DRIVE_ROOT/outputs/gh0st_exhibition_v1_bg_removed/ (mirrors category
# structure). Review the saved images, then copy any good ones over the originals
# in STILLS_BASE to use them in the render.
# ─────────────────────────────────────────────────────────────────────────────

from rembg import remove, new_session
from PIL import Image
import numpy as np

BG_REVIEW_DIR = DRIVE_ROOT / 'outputs' / 'gh0st_exhibition_v1_bg_removed'
BG_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

# ── Thresholds ────────────────────────────────────────────────────────────────
CORNER_DARK_THRESH = 25   # mean pixel value below which corners count as "dark"
CORNER_SIZE        = 24   # px size of corner sample region
BG_MIN_FRACTION    = 0.03 # fraction of pixels that must be removed to save

def corners_are_dark(arr, size=CORNER_SIZE, thresh=CORNER_DARK_THRESH):
    """True if all four corners are already near-black → no bg removal needed."""
    h, w = arr.shape[:2]
    corners = [
        arr[:size,    :size   ],
        arr[:size,    w-size: ],
        arr[h-size:,  :size   ],
        arr[h-size:,  w-size: ],
    ]
    return all(np.mean(c) < thresh for c in corners)


# ── Scan all stills ───────────────────────────────────────────────────────────
all_stills = sorted(STILLS_BASE.rglob('*.png'))
print(f'Scanning {len(all_stills)} stills  (model: u2net_human_seg)')

session = new_session('u2net_human_seg')
saved = skipped_dark = skipped_small = errors = 0

for i, path in enumerate(all_stills):
    try:
        img  = Image.open(path).convert('RGB')
        arr  = np.array(img)

        if corners_are_dark(arr):
            skipped_dark += 1
            continue

        result = remove(img, session=session, post_process_mask=True)  # RGBA
        alpha  = np.array(result.split()[3])
        bg_fraction = (alpha < 128).mean()

        if bg_fraction < BG_MIN_FRACTION:
            skipped_small += 1
            continue

        # Save with black background to preserve exhibition look
        rel      = path.relative_to(STILLS_BASE)
        out_path = BG_REVIEW_DIR / rel
        out_path.parent.mkdir(parents=True, exist_ok=True)
        canvas = Image.new('RGB', result.size, (0, 0, 0))
        canvas.paste(result, mask=result.split()[3])
        canvas.save(out_path)
        saved += 1

    except Exception as e:
        print(f'  ERROR {path.name}: {e}')
        errors += 1

    if i % 20 == 0:
        pct = int(100 * i / len(all_stills))
        print(f'  [{i:3d}/{len(all_stills)}] {pct:3d}%  saved={saved}', end='\r')

print(f'\n')
print(f'Saved to review : {saved}')
print(f'Skipped (dark corners)  : {skipped_dark}')
print(f'Skipped (bg < {BG_MIN_FRACTION:.0%}) : {skipped_small}')
print(f'Errors          : {errors}')
print(f'')
print(f'Review folder   : {BG_REVIEW_DIR}')
print(f'Originals at    : {STILLS_BASE}')
print(f'')
print('To apply: copy any good bg-removed images over the originals in STILLS_BASE.')


In [ ]:
# ── Parse selection file ─────────────────────────────────────────────────────

def stem(name: str) -> str:
    s = Path(name).stem
    return re.sub(r'[\s()]+', '_', s).strip('_')

def parse_selection(path: Path) -> dict:
    categories = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split('/', 1)
        if len(parts) != 2:
            continue
        cat, fname = parts
        categories.setdefault(cat, []).append(fname)
    return categories

selection = parse_selection(SELECTION)

for cat, files in selection.items():
    n_stills = 3 * len(files) - 2
    print(f'  {cat}: {len(files)} styles → {n_stills} stills')


In [ ]:
# ── Build ordered still lists + per-category timing ─────────────────────────

def build_sequence(screen_order: list, selection: dict, base: Path):
    """
    Returns:
      stills      : list of Paths
      cat_timings : list of (frame_index, category_name)
    """
    stills = []
    cat_timings = []

    for i, cat in enumerate(screen_order):
        files  = selection[cat]
        stems  = [stem(f) for f in files]
        cat_dir = base / cat
        cat_start_still = len(stills)

        # First pure still
        stills.append(cat_dir / f'{stems[0]}.png')

        # Within-category: mashup pair + next pure
        for j in range(len(stems) - 1):
            sa, sb = stems[j], stems[j + 1]
            stills.append(cat_dir / f'030{sa}__070{sb}.png')
            stills.append(cat_dir / f'070{sa}__030{sb}.png')
            stills.append(cat_dir / f'{sb}.png')

        # Frame index where this category starts
        frame_start = cat_start_still * (HOLD_FRAMES + MORPH_FRAMES)
        cat_timings.append((frame_start, cat))

        # Boundary to next category
        if i < len(screen_order) - 1:
            next_cat = screen_order[i + 1]
            last_a  = stems[-1]
            first_b = stem(selection[next_cat][0])
            bdir = base / 'boundaries' / f'{cat}_x_{next_cat}'
            stills.append(bdir / f'030{last_a}__070{first_b}.png')
            stills.append(bdir / f'070{last_a}__030{first_b}.png')

    return stills, cat_timings


seq_a, timings_a = build_sequence(SCREEN_A, selection, STILLS_BASE)
seq_b, timings_b = build_sequence(SCREEN_B, selection, STILLS_BASE)

def report(name, seq, cat_timings):
    missing = [p for p in seq if not p.exists()]
    n = len(seq)
    total_frames = n * HOLD_FRAMES + (n - 1) * MORPH_FRAMES
    dur = total_frames / FPS
    print(f'{name}: {n} stills, {len(missing)} missing → {dur:.0f}s ({dur/60:.1f} min)')
    for frame_start, cat in cat_timings:
        print(f'  {frame_start/FPS:6.1f}s  {cat}')
    if missing:
        print(f'  MISSING:')
        for m in missing:
            print(f'    {m.relative_to(STILLS_BASE)}')

report('Screen A', seq_a, timings_a)
print()
report('Screen B', seq_b, timings_b)


In [ ]:
# ── Preview: every 12th still from each screen ────────────────────────────────
from PIL import Image
import IPython.display as ipd

def thumb_strip(seq, step=12, label=''):
    sample = [p for p in seq[::step] if p.exists()]
    if not sample:
        print(f'{label}: no stills found')
        return
    thumbs = [Image.open(p).convert('RGB').resize((128, 128)) for p in sample]
    grid = Image.new('RGB', (len(thumbs) * 128, 128))
    for i, t in enumerate(thumbs):
        grid.paste(t, (i * 128, 0))
    print(label)
    ipd.display(grid)

thumb_strip(seq_a, label='Screen A (every 12th)')
thumb_strip(seq_b, label='Screen B (every 12th)')

In [ ]:
# ── Optical flow helpers ─────────────────────────────────────────────────────
import cv2
import numpy as np
from PIL import Image


def load_img(path: Path, size: tuple) -> np.ndarray:
    img = Image.open(path).convert('RGB')
    w, h = size
    src_w, src_h = img.size
    if abs(src_w / src_h - w / h) > 0.01:
        if src_w / src_h > w / h:
            nw = int(src_h * w / h)
            img = img.crop(((src_w - nw) // 2, 0, (src_w - nw) // 2 + nw, src_h))
        else:
            nh = int(src_w * h / w)
            img = img.crop((0, (src_h - nh) // 2, src_w, (src_h - nh) // 2 + nh))
    return np.array(img.resize((w, h), Image.LANCZOS))


def optical_flow_morph(a: np.ndarray, b: np.ndarray, steps: int) -> list:
    ag = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    bg = cv2.cvtColor(b, cv2.COLOR_RGB2GRAY)
    flow = cv2.calcOpticalFlowFarneback(
        ag, bg, None, pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
    h, w = a.shape[:2]
    xs = np.tile(np.arange(w), (h, 1)).astype(np.float32)
    ys = np.tile(np.arange(h), (w, 1)).T.astype(np.float32)
    frames = []
    for t in np.linspace(0, 1, steps, endpoint=False):
        warped = cv2.remap(a,
            (xs + flow[..., 0] * t).astype(np.float32),
            (ys + flow[..., 1] * t).astype(np.float32),
            cv2.INTER_LINEAR)
        frames.append(cv2.addWeighted(warped, 1 - t, b, t, 0))
    return frames


print('Helpers defined.')


In [ ]:
# ── Render function ──────────────────────────────────────────────────────────
import imageio

def render(seq: list, out_path: Path, label: str) -> float:
    existing = [p for p in seq if p.exists()]
    if not existing:
        raise RuntimeError(f'No stills found at {STILLS_BASE}')
    missing_count = len(seq) - len(existing)
    if missing_count:
        print(f'WARNING: {missing_count} stills missing — skipping them')

    n = len(existing)
    total_frames = n * HOLD_FRAMES + (n - 1) * MORPH_FRAMES
    print(f'Loading {n} stills...')
    imgs = [load_img(p, OUTPUT_SIZE) for p in existing]
    print(f'Loaded. Rendering {total_frames} frames ({total_frames/FPS:.1f}s) → {out_path.name}')
    out_path.parent.mkdir(parents=True, exist_ok=True)

    frame_idx = 0
    with imageio.get_writer(str(out_path), fps=FPS) as writer:
        for i, img in enumerate(imgs):
            # Hold frames
            for _ in range(HOLD_FRAMES):
                writer.append_data(img)
                frame_idx += 1
            # Morph frames
            if i < len(imgs) - 1:
                for morph_frame in optical_flow_morph(img, imgs[i + 1], MORPH_FRAMES):
                    writer.append_data(morph_frame)
                    frame_idx += 1
            if i % 20 == 0:
                print(f'  still {i}/{n}, frame {frame_idx}/{total_frames}', end='\r')

    dur = total_frames / FPS
    print(f'\n✅ {label} → {out_path.name}  ({dur:.0f}s / {dur/60:.1f} min)')
    return dur

print('render() defined.')


In [ ]:
# ── Render Screen A ──────────────────────────────────────────────────────────
dur_a = render(seq_a, OUT_A, 'Screen A')


In [ ]:
# ── Render Screen B ──────────────────────────────────────────────────────────
dur_b = render(seq_b, OUT_B, 'Screen B')


In [ ]:
# ── Sidecar JSON ─────────────────────────────────────────────────────────────
for out_path, dur, screen, seq, cat_timings in [
    (OUT_A, dur_a, 'A', seq_a, timings_a),
    (OUT_B, dur_b, 'B', seq_b, timings_b),
]:
    sidecar = {
        'experiment_id': out_path.stem,
        'screen': screen,
        'duration_seconds': round(dur, 3),
        'fps': FPS,
        'source': 'gh0st_exhibition_v1_flux_lora_v2',
        'n_stills': len(seq),
        'hold_frames': HOLD_FRAMES,
        'morph_frames': MORPH_FRAMES,
        'output_size': OUTPUT_SIZE,
        'category_timings': [
            {'category': cat, 'start_seconds': round(f / FPS, 2)}
            for f, cat in cat_timings
        ],
        'generated_at': datetime.now().isoformat(),
    }
    sp = out_path.with_suffix('.json')
    sp.write_text(json.dumps(sidecar, indent=2))
    print(f'✅ {sp.name}')
    print(json.dumps(sidecar, indent=2))
    print()


In [ ]:
# ── In-notebook preview (Screen A) ───────────────────────────────────────────
from IPython.display import HTML
from base64 import b64encode

data_url = 'data:video/mp4;base64,' + b64encode(open(OUT_A, 'rb').read()).decode()
display(HTML(f'<video width=540 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))